# `ptof_obs_setup_seed`

## What this notebook does
Hand-maintained reference-data seeding and one-off maintenance for the observability pipeline.
It is **not part of the scheduled `obs_fresh_scan` job** -- it's run manually, by a human, when a
new capability needs registering, a threshold needs recording, or a one-time cleanup/backfill is
needed. Every other notebook in this repo *reads* the tables this notebook seeds; nothing else
writes to them.

Rewritten (Phase 7 of the v1.1 simplification pass) to seed each table's **final, current state**
in a single guarded INSERT, instead of the chronological sequence of INSERT/UPDATE/ALTER cells that
built that state up over time. Re-running any cell below is a no-op against rows that already
exist; it does not overwrite hand-curated dispositions on rows already present.

## Position in the pipeline
- **Not in any job DAG.** Run interactively/manually, occasionally, not on a schedule.
- **Downstream readers:** `ptof_obs_latency_detection`, `ptof_obs_mal_output`,
  `ptof_obs_behavioral_correlation`, and `ptof_obs_hallucination_detection` all join against
  `capability_registry` and/or `runtime_allowlist` as ground-truth reference data.
  `ptof_obs_alert.ipynb` reads `threshold_basis` for the "why this number" documentation surfaced
  on Teams cards, and creates/updates `obs_incidents` (this notebook only creates the table once).
  `ptof_obs_weekly_runtime_digest` reads/writes `runtime_observed`.

## Write safety
`runtime_allowlist`, `capability_registry`, and `threshold_basis` are seeded via
`CREATE TABLE IF NOT EXISTS` + a `LEFT ANTI JOIN` insert keyed on each table's natural key --
re-running is always a no-op against rows that already exist and cannot wipe a hand-curated
disposition. `required_fields` is no longer seeded on `capability_registry` inserts (the column
stays in the schema; new rows get NULL).

| Cell | Writes | Idempotent? |
|---|---|---|
| 1 | `capability_registry` | Yes -- `CREATE TABLE IF NOT EXISTS` + anti-join insert on capability |
| 2 | `runtime_allowlist` | Yes -- `CREATE TABLE IF NOT EXISTS` + anti-join insert on (environment, capability) |
| 3 | `obs_incidents` schema | Yes -- `CREATE TABLE IF NOT EXISTS` |
| 4 | `_obs_watermark` | Yes -- `CREATE TABLE IF NOT EXISTS` + `WHERE NOT EXISTS` seed |
| 5 | `threshold_basis` | Yes -- `CREATE TABLE IF NOT EXISTS` + anti-join insert on check_name |
| 6 | `runtime_observed` schema | Yes -- `CREATE TABLE IF NOT EXISTS` |
| 7 | orphaned tables (drop) | Yes -- all `DROP TABLE IF EXISTS`; one-time cleanup kept for the record |

## Tables/views touched
- **Writes:** `runtime_allowlist` (human-curated: which transport/model_config/scheduler_run
  combinations are sanctioned per capability/environment), `capability_registry` (human-curated:
  per-capability metadata -- is it generative, is it GxP-relevant, who owns it, is it active),
  `obs_incidents` (the append-only finding record every detector's alert path writes to --
  created here, populated by `ptof_obs_alert.ipynb`), `_obs_watermark` (incremental-processing
  bookmark for `faithfulness_scores` in the hallucination pipeline), `threshold_basis`
  (documents what every detection threshold in this system is, why it's set where it is, and
  whether it's backed by real evidence yet), `runtime_observed` (automatically-populated log of
  transport/model_config combinations actually seen, kept separate from what's *permitted* in
  `runtime_allowlist`).
- **Reads:** the tables above, for idempotency checks.


In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS + anti-join insert, guarded per capability.
-- Seeds the table's current final state in one shot rather than replaying the chronological
-- history of INSERT/UPDATE edits that built it up. Re-running is a no-op once these rows exist
-- and cannot overwrite a hand-curated disposition edited directly in the table since.
--
-- is_gxp_relevant = true only for SUMMARIZATION capabilities, where every number in the output
-- should trace to an input. Chat, comparative scoring, and projection legitimately emit numbers
-- absent from their prompts, so token-grounding does not apply to them. is_groundable = true for
-- the 4 SAA/ISH capabilities that produce grounded narratives from structured input data; false
-- for dsa_*/probe (computed projections, chat, etc.). dsa_*/probe are active = false (rescoped
-- out of the monitored fleet 2026-09-02 -- rows kept for historical reference).
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.capability_registry (
    capability           STRING,
    is_generative        BOOLEAN,
    is_gxp_relevant      BOOLEAN,
    expected_min_daily   INT,
    required_fields      ARRAY<STRING>,
    owner                STRING,
    active               BOOLEAN,
    notes                STRING,
    silence_grace_hours  INT,
    is_groundable        BOOLEAN
);

INSERT INTO mq_gmdf_dev.oil_obs.capability_registry
  (capability, is_generative, is_gxp_relevant, expected_min_daily, owner, active, notes,
   silence_grace_hours, is_groundable)
SELECT t.* FROM VALUES
  ('dsa_batch_summary',  true,  true,  0,  'dsa-team',  false,
     'vertex26 · summarization — grounding diff applies', NULL, false),
  ('dsa_compare',        true,  false, 0,  'dsa-team',  false,
     'vertex26 · comparative scoring — emits its own rank/score values, grounding N/A', NULL, false),
  ('dsa_copilot',        true,  false, 20, 'dsa-team',  false,
     'vertex26 · conversational chat surface — user_prompt is a turn, not a grounding block', 26, false),
  ('dsa_copilot_step',   true,  false, 0,  'unassigned', false,
     'cortex · registry-drift fix, added 2026-09-01 · observed: test-bot-claude-v1 (605), '
     'vertex26 (59), cortex-claude46 (4), my-agent-triage-rag-ag-grp-dev (2)', NULL, false),
  ('dsa_optimize',       true,  false, 10, 'dsa-team',  false,
     'test-bot-claude-v1 · 100+/100+ failing · emits computed projection deltas, grounding N/A', 2, false),
  ('dsa_session_summary',true,  false, 0,  'unassigned', false,
     'cortex · registry-drift fix, added 2026-09-01 · observed: test-bot-claude-v1 (17), '
     'vertex26 (6)', NULL, false),
  ('probe',              true,  false, 0,  'unassigned', false,
     'cortex · registry-drift fix, added 2026-09-01 · observed: test-bot-claude-v1 (1)', NULL, false),
  ('saa_insight',        true,  true,  5,  'saa-team',  true,
     'cortex · registry-drift fix, added 2026-09-01 · observed: test-bot-claude-v1 (115), '
     'spe-claude-sonnet-46 (34), cortex-claude46 (6), my-agent-triage-rag-ag-grp-dev (1)', 144, true),
  ('sev2_insight',       true,  true,  1,  'saa-team',  true,
     'added 2026-09-02 (rescope) -- 36 rows in audit log, explicit GxP grounding in prompt', 144, true),
  ('summary',            true,  true,  1,  'ish-team',  true,
     'reactivated 2026-09-02 -- test-bot-claude-v1 running daily via saa->ish-eos', 36, true),
  ('watchout_narratives',true,  true,  0,  'ish-team',  true,
     'single call 2026-08-12 · summarization', NULL, true)
AS t(capability, is_generative, is_gxp_relevant, expected_min_daily, owner, active, notes,
     silence_grace_hours, is_groundable)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.capability_registry existing
  ON existing.capability = t.capability;

In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS + anti-join insert, guarded per (environment,
-- capability). Seeds the table's current final state in one shot rather than replaying the
-- chronological history of edits that built it up. Re-running is a no-op once these rows exist.
--
-- runtime_allowlist -- keyed on capability, not scheduler_run.
-- 'live' contains both June's summary on demo-claude-sonnet-4-6-pwc-omi and August's
-- dsa_optimize on test-bot-claude-v1, so scheduler_run does not discriminate.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.runtime_allowlist (
    environment            STRING,
    capability             STRING,
    allowed_transports     ARRAY<STRING>,
    allowed_model_configs  ARRAY<STRING>,
    allowed_scheduler_runs ARRAY<STRING>
);

INSERT INTO mq_gmdf_dev.oil_obs.runtime_allowlist
SELECT t.* FROM VALUES
  ('dev', 'dsa_batch_summary',  array('cortex'), array('vertex26'),                      array('live')),
  ('dev', 'dsa_compare',        array('cortex'), array('vertex26'),                      array('live')),
  ('dev', 'dsa_copilot',        array('cortex'), array('vertex26'),                      array('live')),
  ('dev', 'dsa_copilot_step',   array('cortex'),
          array('test-bot-claude-v1','vertex26','cortex-claude46','my-agent-triage-rag-ag-grp-dev'),
          array('live')),
  ('dev', 'dsa_optimize',       array('cortex'), array('vertex26','test-bot-claude-v1'), array('live')),
  ('dev', 'dsa_session_summary',array('cortex'), array('test-bot-claude-v1','vertex26'), array('live')),
  ('dev', 'probe',              array('cortex'), array('test-bot-claude-v1'),            array('live')),
  ('dev', 'saa_insight',        array('cortex'),
          array('test-bot-claude-v1','spe-claude-sonnet-46','cortex-claude46',
                'my-agent-triage-rag-ag-grp-dev'),
          array('live')),
  ('dev', 'sev2_insight',       array('cortex'),
          array('test-bot-claude-v1','spe-claude-sonnet-46','my-agent-triage-rag-ag-grp-dev'),
          array('live')),
  ('dev', 'summary',            array('cortex'),
          array('mq-ai-dg-poc','ptof-ish-handover-v1-das','demo-claude-sonnet-4-6-pwc-omi',
                'test-bot-claude-v1'),
          array('live','final_handover','pre_handover')),
  ('dev', 'watchout_narratives',array('cortex'), array('demo-claude-sonnet-4-6-pwc-omi'), array('live')),
  ('prod','dsa_batch_summary',  array('cortex'), array('vertex26'), array('live')),
  ('prod','dsa_compare',        array('cortex'), array('vertex26'), array('live')),
  ('prod','dsa_copilot',        array('cortex'), array('vertex26'), array('live')),
  ('prod','dsa_optimize',       array('cortex'), array('vertex26'), array('live')),
  ('prod','summary',            array('cortex'), array('mq-ai-dg-poc','ptof-ish-handover-v1-das'),
                                                 array('final_handover')),
  ('prod','watchout_narratives',array('cortex'), array('vertex26'), array('live'))
AS t(environment, capability, allowed_transports, allowed_model_configs, allowed_scheduler_runs)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.runtime_allowlist existing
  ON existing.environment = t.environment AND existing.capability = t.capability;

In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS. This is the single table every detector's finding
-- ultimately lands in, and the only table ptof_obs_alert.ipynb's Teams-notify query reads from.
--
-- obs_incidents — append-only finding record.
-- MERGE keyed on (detector, source_row_id) so re-detecting the same finding updates rather than
-- duplicates. That gives dedup for free and makes acknowledgement stick across runs.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.obs_incidents (
    detector         STRING,        -- which detector produced this
    source_row_id    STRING,        -- the id being flagged; stable across runs
    capability       STRING,
    severity         STRING,        -- CRITICAL | WARN | INFO
    first_detected   TIMESTAMP,     -- set once, never updated
    last_detected    TIMESTAMP,     -- refreshed each time the finding is still present
    detection_count  BIGINT,        -- how many runs have seen it
    signal_payload   STRING,        -- JSON detail for triage
    notified_at      TIMESTAMP,     -- stamped when the alert reports it
    acknowledged_by  STRING,        -- set by a human, by hand
    acknowledged_at  TIMESTAMP,
    resolved_at      TIMESTAMP      -- set by hand once remediated
) CLUSTER BY (first_detected, detector);

In [ ]:
%sql
-- SAFE to re-run: table creation is IF NOT EXISTS, and the seed insert below is itself
-- WHERE-NOT-EXISTS-guarded. This is the incremental-processing bookmark that lets
-- ptof_obs_hallucination_detection's faithfulness_scores compute only new rows each run instead
-- of rescanning all of v_llm_bronze -- the one detector in this codebase using the
-- watermark+MERGE incremental pattern.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs._obs_watermark (
    detector          STRING,
    last_processed_ts TIMESTAMP,
    updated_at        TIMESTAMP
);

-- Seed only if absent. INSERT OVERWRITE rewinds the watermark 24h on every seed run, silently
-- re-scoring a day of rows and masking whether task 04's advance cell is working.
INSERT INTO mq_gmdf_dev.oil_obs._obs_watermark
SELECT 'faithfulness_scores', current_timestamp() - INTERVAL 24 HOURS, current_timestamp()
WHERE NOT EXISTS (SELECT 1 FROM mq_gmdf_dev.oil_obs._obs_watermark
                  WHERE detector = 'faithfulness_scores');

In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS + anti-join insert, guarded per check_name. Seeds
-- the table's current final state in one shot rather than replaying the chronological history
-- of INSERT cells that built it up across the v1.1 simplification and architecture-hardening
-- passes. Re-running is a no-op once these rows exist.
--
-- threshold_basis — what each threshold is, why, and whether evidence supports it.
-- Every number in this pipeline is provisional: there are no labelled examples yet, so most are
-- anchored to observed ranges rather than to confirmed incidents.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.threshold_basis (
    check_name STRING,
    threshold  STRING,
    basis      STRING,
    set_on     STRING,
    status     STRING
);

INSERT INTO mq_gmdf_dev.oil_obs.threshold_basis
SELECT t.* FROM VALUES
  ('capability_error_rate', 'error_rate = 1.0 & n>=5, or > 0.20 & n>=10',
   'observed 0.00 or 0.93-1.00; no middle ground in data', '2026-08-20', 'provisional'),
  ('capability_error_rate_sustained', '0.50',
   'GxP-only fleet (SAA/ISH rescope 2026-09-02). Bimodal failure distribution (0% or 93%+). '
   '50% = majority-failing gate. Observed data supports no noise increase at this level.',
   '2026-09-03', 'provisional'),
  ('capability_silence dsa_copilot', 'silence_grace_hours = 26',
   'raised from 18 after two false breaches (19.5h, 22.7h), zero true', '2026-08-20', 'deprecated'),
  ('capability_silence dsa_optimize', 'silence_grace_hours = 2',
   'automated hourly poll; 2h silence unambiguous', '2026-08-20', 'deprecated'),
  ('hallucination medium', 'pctile < 0.05 AND similarity < 0.75',
   'similarity band 0.692-0.887 over 136 rows; floor excludes single-row capabilities',
   '2026-08-20', 'unvalidated - no row qualifies'),
  ('hallucination_similarity_cross_check', 'resp_vs_prompt_similarity < 0.80',
   'Cross-check gate on the ungrounded-token-count escalation path (Rule 3) in '
   'hallucination_signal: requires both the regex-based ungrounded-token signal AND semantic '
   'similarity to agree something is off before escalating to hallucination_risk = high, so a '
   'single noisy signal cannot escalate alone. NULL similarity (scoring unavailable) is '
   'fail-open by design, so it also satisfies this branch rather than suppressing the finding.',
   '2026-09-05', 'provisional'),
  ('handover_delivery_rate', 'failure_pct_7d > 20 AND attempts >= 10',
   'all-time baseline 9.3% (15 of 162 since Jun 19); currently 12.5% over 7d',
   '2026-08-20', 'provisional'),
  ('latency_anomaly', 'p95 + 3*IQR, is_reliable only',
   'no capability qualifies yet (needs n>=30 AND span>=7d); live ~2026-08-25',
   '2026-08-20', 'not yet active'),
  ('latency_anomaly_findings_min_count', '>= 2 anomalous calls in the detection window',
   'Aggregation guard on latency_anomaly_findings, keyed on (capability, verdict): a single '
   'slow call is noise at the observed 5-15 calls/hour volume for SAA/ISH capabilities, so at '
   'least 2 anomalous calls in the same window are required before a finding is produced.',
   '2026-09-04', 'provisional'),
  ('latency_baseline_n_samples', '>= 30',
   'Guard on capability_latency_baseline reliability. Learned the hard way: dsa_batch_summary '
   'had n=1, so p50=p95=p99=bound and a 2826ms call fired as an anomaly against a baseline that '
   'was really just one prior data point.',
   '2026-09-05', 'provisional'),
  ('latency_baseline_span_days', '>= 6 distinct calendar days with a call',
   'Guard on capability_latency_baseline reliability, paired with n_samples >= 30. Counts '
   'distinct calendar days with a call, not calendar range end-to-end, so a single stale early '
   'call cannot stretch the window into reliable without real day-over-day volume. Floor is 6 '
   '(not 7) specifically to preserve saa_insight, which sits at exactly 6 distinct days and was '
   'already is_reliable=true under the prior check.',
   '2026-09-05', 'provisional'),
  ('long_running_incident', 'first_detected <= now() - INTERVAL 3 DAYS',
   'Fix (v1.1a, A.4): switched from detection_count >= 20 AND age >= 24h to age-based. '
   'detection_count counts RUNS that saw it, not occurrences -- at 5-min triggering the old '
   'threshold tripped within 2h of any unacknowledged incident, meaning nothing. Three days '
   'unacknowledged is a real signal.',
   '2026-09-08', 'revised'),
  ('prompt_size_drift_multiplier', '2x nightly baseline p95 (prompt or response chars)',
   'Leading indicator for latency anomalies: a prompt-template regression (embedding full shift '
   'history, wrong model config, copy-paste error) shows up here before it crosses the latency '
   'threshold, giving time to catch a deployment regression within the hour. 2x is a round-number '
   'margin over baseline p95, not yet calibrated against a labelled incident.',
   '2026-09-04', 'provisional'),
  ('rapid_human_correction',
   'AI publish and ISH correction match on (shift_date, shift_type, batch_nbr) within 10 minutes',
   'dsa_* capabilities write NULL or an empty string for shift_date/shift_type/batch_nbr, so '
   'ai_publish excludes them entirely -- 0 rows is expected, not a failure. Revisit after '
   '2026-10-01; ptof_obs_verification.ipynb WARNs if still 0 rows past that date.',
   '2026-09-01', 'not yet active'),
  ('response_baseline_min_rows', '>= 20 eligible rows per capability',
   'Guard shared by response_schema_baseline and response_field_baseline (nightly baseline '
   'notebook): a capability with fewer than 20 eligible (success, non-blank, non-fastfail) rows '
   'in the lookback window is excluded from the baseline rather than inferring a schema shape '
   'from too few samples.',
   '2026-09-05', 'provisional'),
  ('runtime_observed_digest_floor', 'occurrences_7d >= 5',
   'Suppresses a single stray call from the digest body. Explicitly NOT an auto-approval '
   'threshold: occurrence count does not discriminate sanctioned from misconfigured. '
   'dsa_optimize at 0/121 would clear any floor within a day.',
   '2026-08-27', 'unvalidated'),
  ('runtime_violation_digest_cadence', 'weekly, Monday 08:00 America/Indianapolis',
   'Unlisted capabilities are normal in ISH: the allowlist is maintained more slowly than the '
   'agent ships. Replaces 529 CRITICAL cards for one configuration.',
   '2026-08-27', 'provisional'),
  ('runtime_violation_immediate', 'transport OR model_config sanctioned for no capability in env',
   'A new capability on an already-sanctioned transport + model_config is a paperwork lag. A new '
   'transport or model config is a different claim about the system. Tier split, not a numeric '
   'threshold. scheduler_run excluded: live spans two agent generations and does not discriminate.',
   '2026-08-27', 'provisional'),
  ('ungrounded_token_count', '> 3 AND is_gxp_relevant',
   'only 3 low-volume capabilities are gxp_relevant; untested where it applies',
   '2026-08-20', 'unvalidated'),
  ('write_lag', 'p95_ingest_only_s > 10',
   'observed 2.1-2.3s across all hours; ~4x headroom', '2026-08-20', 'provisional')
AS t(check_name, threshold, basis, set_on, status)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.threshold_basis existing
  ON existing.check_name = t.check_name;

In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS, schema only. This is the automatically-populated
-- log of transport/model_config combinations actually observed in traffic, deliberately kept
-- separate from runtime_allowlist (which stays 100% human-curated) so "what we saw" and "what we
-- decided to permit" never get conflated. ptof_obs_weekly_runtime_digest.ipynb is what writes
-- rows here and reads digest_reported_at/disposition back out.
--
-- runtime_observed — a record of what has been SEEN, kept separate from what is PERMITTED.
-- runtime_allowlist stays human-seeded. This table is written automatically; disposition is not.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.runtime_observed (
    environment          STRING,
    violation_signature  STRING,
    capability           STRING,
    transport            STRING,
    model_config         STRING,
    scheduler_run        STRING,
    violation_type       STRING,
    violation_tier       STRING,
    first_seen           TIMESTAMP,
    last_seen            TIMESTAMP,
    occurrences_7d       BIGINT,
    digest_reported_at   TIMESTAMP,
    disposition          STRING,   -- NULL = undispositioned | sanctioned | rejected | transient
    dispositioned_by     STRING,
    dispositioned_at     TIMESTAMP
);

In [ ]:
%sql
-- ONE-TIME cleanup, safe to re-run only because all DROPs are IF EXISTS (a re-run is a harmless
-- no-op, not because dropping tables is generally idempotent-safe). Removes tables orphaned by
-- the v1.1 pipeline simplification pass: hallucination_verdicts (Layer 1 dead code, never had
-- verify_grounding data flow through it), blank_output_incidents and transport_violations
-- (collapsed into CTEs in ptof_obs_mal_output.ipynb), capability_outage_findings (duplicate of
-- capability_error_rate_sustained), response_schema_baseline (superseded by
-- response_field_baseline, computed nightly), plus two older orphans (transport_allowlist,
-- success_rate_daily) kept here for the record.
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.hallucination_verdicts;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.blank_output_incidents;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.transport_violations;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.capability_outage_findings;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.response_schema_baseline;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.transport_allowlist;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.success_rate_daily;